# Not All Factors Crowd Equally: Modeling, Measuring, and Trading on Alpha Decay

## Paper Citation
Lee, Chorok. "Not All Factors Crowd Equally: Modeling, Measuring, and Trading on Alpha Decay." arXiv preprint arXiv:2512.11913 (2025).

## Strategy Description
This notebook implements a quantitative trading strategy based on the paper "Not All Factors Crowd Equally: Modeling, Measuring, and Trading on Alpha Decay" by Chorok Lee. The strategy focuses on trading factors that exhibit hyperbolic decay in their alpha, particularly momentum and reversal factors. The strategy aims to capture the decay in factor alpha over time and manage crowding risk.

## Abstract
We derive a specific functional form for factor alpha decay -- hyperbolic decay alpha(t) = K/(1+lambda*t) -- from a game-theoretic equilibrium model, and test it against linear and exponential alternatives. Using eight Fama-French factors (1963--2024), we find: (1) Hyperbolic decay fits mechanical factors. Momentum exhibits clear hyperbolic decay (R^2 = 0.65), outperforming linear (0.51) and exponential (0.61) baselines -- validating the equilibrium foundation. (2) Not all factors crowd equally. Mechanical factors (momentum, reversal) fit the model; judgment-based factors (value, quality) do not -- consistent with a signal-ambiguity taxonomy paralleling Hua and Sun's "barriers to entry." (3) Crowding accelerated post-2015. Out-of-sample, the model over-predicts remaining alpha (0.30 vs. 0.15), correlating with factor ETF growth (rho = -0.63). (4) Average returns are efficiently priced. Crowding-based factor selection fails to generate alpha (Sharpe: 0.22 vs. 0.39 factor momentum benchmark). (5) Crowding predicts tail risk. Out-of-sample (2001--2024), crowded reversal factors show 1.7--1.8x higher crash probability (bottom decile returns), while crowded momentum shows lower crash risk (0.38x, p = 0.006). Our findings extend equilibrium crowding models (DeMiguel et al.) to temporal dynamics and show that crowding predicts crashes, not means -- useful for risk management, not alpha generation.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'FB']
HYPERBOLIC_DECAY_PARAMS = {'K': 1.0, 'lambda': 0.05}
RISK_FREE_RATE = 0.02
# Hypothesis: Trading on hyperbolic decay of momentum and reversal factors can generate alpha and manage crowding risk.

## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start='2000-01-01', end='2024-12-31', group_by='ticker')

# Compute momentum and reversal factors
def compute_factors(data):
    momentum = data['Adj Close'].pct_change(21)
    reversal = data['Adj Close'].pct_change(1).rolling(window=21).mean()
    return momentum, reversal

momentum, reversal = compute_factors(data)

# Cross-sectional normalization
def normalize_factors(factors):
    return (factors - factors.mean()) / factors.std()

momentum_normalized = normalize_factors(momentum)
reversal_normalized = normalize_factors(reversal)

## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Signal generation
def generate_signals(momentum, reversal):
    signals = momentum_normalized + reversal_normalized
    return signals

signals = generate_signals(momentum_normalized, reversal_normalized)

# Position sizing
def position_sizing(signals):
    positions = signals.rank(axis=1, pct=True)
    return positions

positions = position_sizing(signals)

## Phase 4 — Vectorized Backtest

In [ ]:
# Vectorized backtest
def backtest(data, positions):
    returns = data['Adj Close'].pct_change().shift(-1)
    portfolio_returns = (returns * positions).sum(axis=1)
    cumulative_returns = (1 + portfolio_returns).cumprod()
    return cumulative_returns

cumulative_returns = backtest(data, positions)

## Phase 5 — Performance Metrics

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Performance metrics
def performance_metrics(cumulative_returns):
    sharpe = np.mean(cumulative_returns) / np.std(cumulative_returns)
    sortino = np.mean(cumulative_returns) / np.std(cumulative_returns[cumulative_returns < 0])
    calmar = np.mean(cumulative_returns) / (1 - np.min(cumulative_returns))
    max_drawdown = (1 - np.min(cumulative_returns)) * 100
    
    print(f'Sharpe Ratio: {sharpe:.2f}')
    print(f'Sortino Ratio: {sortino:.2f}')
    print(f'Calmar Ratio: {calmar:.2f}')
    print(f'Max Drawdown: {max_drawdown:.2f}%%')
    
    # Plot equity curve
    plt.plot(cumulative_returns)
    plt.title('Equity Curve')
    plt.xlabel('Time')
    plt.ylabel('Cumulative Returns')
    plt.show()

performance_metrics(cumulative_returns)

## Phase 6 — Monitoring Stub

In [ ]:
# Monitoring stub
def monitor(data, positions):
    current_prices = data['Adj Close'].iloc[-1]
    current_positions = positions.iloc[-1]
    daily_pnl = (current_prices * current_positions).sum()
    print(f'Daily P&L: {daily_pnl:.2f}')
    print('Current Positions:')
    print(current_positions)

monitor(data, positions)